Import Libraries

In [ ]:
import jax
import flax
import optax
from jax import lax, random, numpy as jnp
from jax import random, grad, vmap, hessian, jacfwd, jit
from jax import config
from flax import linen as nn

import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors

import os
from tqdm.auto import tqdm

#os.environ['CUDA_VISIBLE_DEVICES'] = '1'
config.update("jax_enable_x64", True)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true' # default is true, 90% of GPU VRAM preallocated


In [ ]:
n_gpu = len(jax.devices())
jax.devices()

In [ ]:
# For PC
DATA_DIR = Path.cwd().parent.parent / "data"

In [ ]:
DATA_FILE = DATA_DIR / "data_T20_S30_G50_reduced.dat"

dat = np.loadtxt(DATA_FILE, skiprows=1, delimiter=',') # data on CPU
print (dat.shape)

# swap header for non reduced dataset as it was not labelled properly
if DATA_FILE == DATA_DIR / "data_T20_S30_G50.dat":
    dat[:, [0, 1]] = dat[:, [1, 0]]

# crop to focus on area of interest in the grid
dat = dat[(dat[:,1] > 150) & (dat[:,1] < 350)] # selects all rows where the y-coordinate falls strictly between 150 and 350

# Calculate dimensions
nx = len(np.unique(dat[:, 0])) # take every row in x col 
ny = len(np.unique(dat[:, 1])) # take every row in y col

print(dat.shape)
print(f"Grid dimensions: nx={nx}, ny={ny}")
print(np.unique(dat[:,0]).shape, np.unique(dat[:,1]).shape) # unique coordinate values along each individual axis

x_l, x_u, y_l, y_u = np.unique(dat[:,0]).min(), np.unique(dat[:,0]).max(), np.unique(dat[:,1]).min(), np.unique(dat[:,1]).max()
ext = [x_l, x_u, y_l, y_u] # plot boundary
print(ext)

In [ ]:
"""
Plot the system with boundaries as the perimeter of the plot
"""
# Source
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1,2,1)
s_plot = dat[:,2].reshape(nx, ny).T
mesh2 = plt.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
plt.colorbar(mesh2); plt.xlabel('x'); plt.ylabel('y')
plt.title('Source', fontsize='x-large')

# Solution
ax1 = fig.add_subplot(1,2,2)
s_plot = dat[:,3].reshape(nx, ny).T
mesh2 = plt.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
plt.colorbar(mesh2); plt.xlabel('x'); plt.ylabel('y')
plt.title('Solution', fontsize='x-large')

In [ ]:
dat = jnp.array(dat) # data now on GPU
x, y, s, u_sim = dat[:,0].reshape(-1, 1), dat[:,1].reshape(-1, 1), dat[:,2].reshape(-1, 1), dat[:,3].reshape(-1, 1)

# Neumann Boundaries (Top and Bottom edges)
bc_n = ((y == y_l) | (y == y_u)).flatten()
print('Neumann BC: #sample = %d'%bc_n.sum())

# Boundary Conditions (bc), Periodic upper (pu) and Periodic  lower (pl)
bc_pl, bc_pu = ((x == x_l)).flatten(), ((x == x_u)).flatten()
print('Periodic BC: #sample = %d %d'%(bc_pl.sum(), bc_pu.sum()))

# bc_d = ((y == y_l) | (y == y_u) | (x == x_l) | (x == x_u)).flatten()
# print ('Dirichlet BC: #sample = %d'%bc_d.sum())

# Boundary points for mini-batching
x_n, y_n = x[bc_n], y[bc_n]
x_pl, y_pl = x[bc_pl], y[bc_pl]
x_pu, y_pu = x[bc_pu], y[bc_pu]

In [ ]:
# Network Hyperparameters
M = 256 
hidden_layers = [128, 128, 128, 128] 
total_features = (2 * M) + sum(hidden_layers) # 512 + 512 = 1024

# Base Random Matrix
rff_key = jax.random.PRNGKey(99)
B_matrix_base = jax.random.normal(rff_key, (2, M)) 

class FeatureExtractor(nn.Module):
    hidden_layers: list
    B_matrix_base: jnp.ndarray 
    
    @nn.compact
    def __call__(self, x_in, y_in):
        sigma = 25.0 
        
        # We keep Tikhonov trainable because it is convex and safe to meta-learn
        _ = self.param('raw_tik', nn.initializers.constant(-5.0), (1,))
        
        # Spatial Normalization
        x_norm = 2.0 * (x_in - x_l) / (x_u - x_l) - 1.0
        y_norm = 2.0 * (y_in - y_l) / (y_u - y_l) - 1.0
        v = jnp.concatenate([x_norm, y_norm], axis=-1)

        # Apply the fixed massive sigma
        proj = 2.0 * jnp.pi * jnp.dot(v, self.B_matrix_base * sigma)
        h_rff = jnp.concatenate([jnp.sin(proj), jnp.cos(proj)], axis=-1)
        all_features = [h_rff]

        h = h_rff
        for size in self.hidden_layers:
            h = nn.Dense(size, kernel_init=nn.initializers.he_normal())(h)
            h = nn.tanh(h) 
            all_features.append(h)
            
        return jnp.concatenate(all_features, axis=-1)
    
model = FeatureExtractor(
    hidden_layers=hidden_layers, 
    B_matrix_base=B_matrix_base 
)

# Initialize network parameters
key = jax.random.PRNGKey(0)
dummy_x = jnp.array([0.0])
dummy_y = jnp.array([0.0])
params = model.init(key, dummy_x, dummy_y)

# (Keep your get_f and get_f_dir functions exactly as they were here)
def get_f(params, x_val, y_val):
    return model.apply(params, x_val, y_val)

def get_f_dir(params, x_val, y_val):
    f_x = jacfwd(get_f, argnums=1)(params, x_val, y_val)
    f_y = jacfwd(get_f, argnums=2)(params, x_val, y_val)
    f_xx = jacfwd(jacfwd(get_f, argnums=1), argnums=1)(params, x_val, y_val)
    f_yy = jacfwd(jacfwd(get_f, argnums=2), argnums=2)(params, x_val, y_val)
    
    f_x = jnp.squeeze(f_x, axis=-1)
    f_y = jnp.squeeze(f_y, axis=-1)
    f_xx = jnp.squeeze(f_xx, axis=(-1, -2))
    f_yy = jnp.squeeze(f_yy, axis=(-1, -2))
    return f_x, f_y, f_xx, f_yy

f_spatial_vmap = vmap(get_f, in_axes=(None, 0, 0))
f_dir_spatial_vmap = vmap(get_f_dir, in_axes=(None, 0, 0))

In [ ]:
chunk_size = 256

def get_f_chunk(params, x_val, y_val, start_idx, end_idx):
    f = model.apply(params, x_val, y_val)
    return lax.dynamic_slice(f, (start_idx,), (end_idx - start_idx,))

def get_f_dir_chunk(params, x_val, y_val, start_idx, end_idx):
    # f_y is needed for the Neumann Boundaries
    f_y = jacfwd(get_f_chunk, argnums=2)(params, x_val, y_val, start_idx, end_idx)
    # f_xx and f_yy are needed for the PDE
    f_xx = jacfwd(jacfwd(get_f_chunk, argnums=1), argnums=1)(params, x_val, y_val, start_idx, end_idx)
    f_yy = jacfwd(jacfwd(get_f_chunk, argnums=2), argnums=2)(params, x_val, y_val, start_idx, end_idx)
    
    # Squeeze out the inner gradient dimensions to match expected shapes
    f_y = jnp.squeeze(f_y, axis=-1)
    f_xx = jnp.squeeze(f_xx, axis=(-1, -2))
    f_yy = jnp.squeeze(f_yy, axis=(-1, -2))
    
    return f_y, f_xx, f_yy

f_chunk_spatial_vmap = vmap(get_f_chunk, in_axes=(None, 0, 0, None, None))
f_dir_chunk_spatial_vmap = vmap(get_f_dir_chunk, in_axes=(None, 0, 0, None, None))

In [ ]:
@jax.jit(static_argnames=['total_features', 'chunk_size'])
def update_direct_solve(params, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size, tik_reg=1e-4):
    A_chunks = []
    
    for start_idx in range(0, total_features, chunk_size):
        end_idx = min(start_idx + chunk_size, total_features)
        
        # 1. PDE Interior (Scaled down by 100 to match your original dataset logic)
        _, f_xx, f_yy = f_dir_chunk_spatial_vmap(params, x_pde, y_pde, start_idx, end_idx)
        A_pde_chunk = (f_xx + f_yy) / 100.0
        
        # 2. Neumann BC (f_y = 0)
        f_y_n, _, _ = f_dir_chunk_spatial_vmap(params, x_n, y_n, start_idx, end_idx)
        A_nbc_chunk = f_y_n
        
        # 3. Periodic BC (f_pl - f_pu = 0)
        f_pl = f_chunk_spatial_vmap(params, x_pl, y_pl, start_idx, end_idx)
        f_pu = f_chunk_spatial_vmap(params, x_pu, y_pu, start_idx, end_idx)
        A_pbc_chunk = f_pl - f_pu
        
        # Stack this feature chunk vertically
        A_chunk = jnp.vstack([A_pde_chunk, A_nbc_chunk, A_pbc_chunk])
        A_chunks.append(A_chunk)
        
    # Combine all feature chunks horizontally to get the full A matrix
    A = jnp.hstack(A_chunks)
    
    # Build the b vector
    b_pde = s_pde / 100.0
    b_nbc = jnp.zeros((x_n.shape[0], 1))
    b_pbc = jnp.zeros((x_pl.shape[0], 1))
    b = jnp.vstack([b_pde, b_nbc, b_pbc])
    
    # Solve w using Tikhonov Regularized Normal Equations
    AtA = A.T @ A + tik_reg * jnp.eye(total_features)
    Atb = A.T @ b
    
    w_new = jnp.linalg.solve(AtA, Atb)
    
    return w_new

In [ ]:
# Function to get the final predicted u using the extracted w weights
def get_u(params, x_val, y_val, w_current):
    f = model.apply(params, x_val, y_val)
    return jnp.dot(f, w_current) 

def get_u_dir(params, x_val, y_val, w_current):
    u_y = jacfwd(get_u, argnums=2)(params, x_val, y_val, w_current)
    u_xx = jacfwd(jacfwd(get_u, argnums=1), argnums=1)(params, x_val, y_val, w_current)
    u_yy = jacfwd(jacfwd(get_u, argnums=2), argnums=2)(params, x_val, y_val, w_current)
    
    u_y = jnp.squeeze(u_y)
    u_xx = jnp.squeeze(u_xx)
    u_yy = jnp.squeeze(u_yy)
    
    return u_y, u_xx, u_yy

u_spatial_vmap = vmap(get_u, in_axes=(None, 0, 0, None))
u_dir_spatial_vmap = vmap(get_u_dir, in_axes=(None, 0, 0, None))

def full_forward_and_loss(params, x_pde, y_pde, s_pde, u_sim_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size):
    
    raw_tik = params['params']['raw_tik'][0]
    tik_reg = 10.0 ** raw_tik 
    
    w_current = update_direct_solve(params, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size, tik_reg)
    
    f_pde = f_spatial_vmap(params, x_pde, y_pde)
    u_pred = jnp.dot(f_pde, w_current)
    loss_data = jnp.mean((u_pred - jnp.squeeze(u_sim_pde)) ** 2)
    
    u_y_n, _, _ = u_dir_spatial_vmap(params, x_n, y_n, w_current)
    loss_n = jnp.mean(u_y_n ** 2)
    
    u_pl = jnp.squeeze(u_spatial_vmap(params, x_pl, y_pl, w_current))
    u_pu = jnp.squeeze(u_spatial_vmap(params, x_pu, y_pu, w_current))
    loss_p = jnp.mean((u_pl - u_pu) ** 2)
    
    total_loss = loss_data + loss_n + loss_p
    
    return total_loss, (w_current, tik_reg) 

loss_grad_fn = jax.jit(
    jax.value_and_grad(full_forward_and_loss, argnums=0, has_aux=True), 
    static_argnames=['total_features', 'chunk_size']
)

optimizer = optax.adamw(learning_rate=1e-4, weight_decay=1e-4)

@jax.jit(static_argnames=['total_features', 'chunk_size'])
def update_network(params, opt_state, x_pde, y_pde, s_pde, u_sim_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size):
    (loss, (w_current, current_tik)), grads = loss_grad_fn(
        params, x_pde, y_pde, s_pde, u_sim_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size
    )
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    new_params = optax.apply_updates(params, updates)
    
    return new_params, opt_state, loss, w_current, current_tik

In [ ]:
# Sampling 1000 points instead of full grid

# # Training Hyperparameters
# epochs = 10
# spatial_batch_size = 1000 
# bc_batch_size = 250 # We will use this as a maximum cap now

# # Initialize weights and optimizer state
# opt_state = optimizer.init(params)
# w_final = None 

# total_points = x.shape[0]
# total_n_points = x_n.shape[0]
# total_p_points = x_pl.shape[0]

# # Dynamically cap the boundary batch sizes so they never exceed the available points
# n_batch_size = min(bc_batch_size, total_n_points)
# p_batch_size = min(bc_batch_size, total_p_points)

# rng = jax.random.PRNGKey(42)

# for epoch in tqdm(range(epochs), desc="Training Phase"):
#     rng, pde_key, n_key, p_key = jax.random.split(rng, 4)
    
#     # 1. Sample Interior PDE points
#     pde_idx = jax.random.choice(pde_key, total_points, shape=(spatial_batch_size,), replace=False)
#     x_pde_chunk = x[pde_idx]
#     y_pde_chunk = y[pde_idx]
#     s_pde_chunk = s[pde_idx]
    
#     # 2. Sample Neumann Boundary points (Using the capped n_batch_size)
#     n_idx = jax.random.choice(n_key, total_n_points, shape=(n_batch_size,), replace=False)
#     x_n_chunk = x_n[n_idx]
#     y_n_chunk = y_n[n_idx]
    
#     # 3. Sample Periodic Boundary points (Using the capped p_batch_size)
#     p_idx = jax.random.choice(p_key, total_p_points, shape=(p_batch_size,), replace=False)
#     x_pl_chunk = x_pl[p_idx]
#     y_pl_chunk = y_pl[p_idx]
#     x_pu_chunk = x_pu[p_idx] 
#     y_pu_chunk = y_pu[p_idx] 
    
#     # 4. Perform the massive Forward Pass, Direct Solve, and Backprop step
#     params, opt_state, loss, w_current = update_network(
#         params, opt_state, 
#         x_pde_chunk, y_pde_chunk, s_pde_chunk, 
#         x_n_chunk, y_n_chunk, 
#         x_pl_chunk, y_pl_chunk, x_pu_chunk, y_pu_chunk, 
#         total_features, chunk_size
#     )
    
#     w_final = w_current
    
#     if epoch % 1 == 0:
#         tqdm.write(f"Epoch {epoch:03d} | Total Loss: {loss:.4e}")

In [ ]:
# Plotting for above cell

# chunk_size_eval = 10000
# u_pred_list = []

# print("Evaluating full grid...")
# for i in range(0, x.shape[0], chunk_size_eval):
#     x_c = x[i : i+chunk_size_eval]
#     y_c = y[i : i+chunk_size_eval]
    
#     # Get features for this chunk using the fully trained parameters
#     f_chunk = f_spatial_vmap(params, x_c, y_c)
    
#     # Multiply by the final linear weights from the last epoch
#     u_chunk = jnp.dot(f_chunk, w_final)
#     u_pred_list.append(u_chunk)

# # Stack all chunks back together
# u_pred = jnp.vstack(u_pred_list)

# # 2. Calculate final global metrics
# mse = jnp.mean((u_sim - u_pred)**2)
# mae = jnp.mean(jnp.abs(u_sim - u_pred))
# rl2 = jnp.linalg.norm(u_sim - u_pred) / jnp.linalg.norm(u_sim)

# print(f"Final Metrics -> MSE: {mse:.2e} | MAE: {mae:.2e} | RL2: {rl2:.2e}")

# # 3. Reshape for Matplotlib (nx and ny were defined in your Phase 1 cell)
# s_plot = s.reshape(nx, ny).T
# u_sim_plot = u_sim.reshape(nx, ny).T
# u_pred_plot = u_pred.reshape(nx, ny).T

# # 4. Plotting
# fig = plt.figure(figsize=(18, 5))

# # Source
# ax1 = fig.add_subplot(1, 3, 1)
# mesh1 = ax1.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
# fig.colorbar(mesh1, ax=ax1)
# ax1.set_xlabel('x')
# ax1.set_ylabel('y')
# ax1.set_title('Source (s)', fontsize='x-large')

# # Simulation Ground Truth
# ax2 = fig.add_subplot(1, 3, 2)
# mesh2 = ax2.imshow(u_sim_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
# fig.colorbar(mesh2, ax=ax2)
# ax2.set_xlabel('x')
# ax2.set_ylabel('y')
# ax2.set_title('Simulation Ground Truth (u)', fontsize='x-large')

# # PINN Prediction
# ax3 = fig.add_subplot(1, 3, 3)
# mesh3 = ax3.imshow(u_pred_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
# fig.colorbar(mesh3, ax=ax3)
# ax3.set_xlabel('x')
# ax3.set_ylabel('y')
# ax3.set_title('PINN Prediction', fontsize='x-large')

# plt.tight_layout()
# plt.show()

In [ ]:
# Training Hyperparameters
epochs = 500 

# Initialize weights and optimizer state
opt_state = optimizer.init(params)
w_final = None 

print(f"Commencing Supervised Global Training: {x.shape[0]} points per epoch.")

for epoch in tqdm(range(epochs), desc="Global Training Phase"):
    
    # Pass the ENTIRE grid into the update step, including u_sim!
    params, opt_state, loss, w_current, curr_tik = update_network(
        params, opt_state, 
        x, y, s, u_sim,              
        x_n, y_n,                    
        x_pl, y_pl, x_pu, y_pu,      
        total_features, chunk_size
    )
    
    w_final = w_current
    
    if epoch % 5 == 0:
        tqdm.write(f"Epoch {epoch:03d} | Loss: {loss:.4e} | Tik Reg: {curr_tik:.2e}")

# Phase 4: Full Grid Evaluation and Plotting
print("Evaluating final predictions...")

f_final = f_spatial_vmap(params, x, y)
u_pred = jnp.dot(f_final, w_final)

# Calculate final global metrics
mse = jnp.mean((u_sim - u_pred)**2)
mae = jnp.mean(jnp.abs(u_sim - u_pred))
rl2 = jnp.linalg.norm(u_sim - u_pred) / jnp.linalg.norm(u_sim)

print(f"Final Metrics -> MSE: {mse:.2e} | MAE: {mae:.2e} | RL2: {rl2:.2e}")

# Reshape for Matplotlib
s_plot = s.reshape(nx, ny).T
u_sim_plot = u_sim.reshape(nx, ny).T
u_pred_plot = u_pred.reshape(nx, ny).T

# Plotting
fig = plt.figure(figsize=(18, 5))

ax1 = fig.add_subplot(1, 3, 1)
mesh1 = ax1.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh1, ax=ax1)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Source (s)', fontsize='x-large')

ax2 = fig.add_subplot(1, 3, 2)
mesh2 = ax2.imshow(u_sim_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh2, ax=ax2)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Simulation Ground Truth (u)', fontsize='x-large')

ax3 = fig.add_subplot(1, 3, 3)
mesh3 = ax3.imshow(u_pred_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh3, ax=ax3)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('Supervised PINN', fontsize='x-large')

plt.tight_layout()
plt.show()

In [ ]:
# Ideal: SSR = 2.38e+01 | MSE = 8.65e-02 | MAE = 2.35e-01 | RL2 = 2.90e-01

In [ ]:
print(jax.nn.softplus(params['params']['raw_sigma'][0]))